# 02 — PDF to Page Images
Convert every page of every PDF in `input_folder` to a PNG in `temp_images/`.

Output naming: `{stem}_page_{n}.png`  (1-indexed)

In [ ]:
%pip install pymupdf boto3 Pillow reportlab -q

In [ ]:
import fitz
import os
from pathlib import Path

from utils import get_logger
logger = get_logger("02_pdf_to_images")

BASE_DIR     = Path(os.getcwd()).parent
INPUT_FOLDER = BASE_DIR / "input_folder"
TEMP_FOLDER  = BASE_DIR / "temp_images"
PAGE_DPI     = 150

TEMP_FOLDER.mkdir(exist_ok=True)

logger.info("Input  : %s", INPUT_FOLDER)
logger.info("Temp   : %s", TEMP_FOLDER)
logger.info("DPI    : %d", PAGE_DPI)

In [ ]:
def pdf_to_images(pdf_path: Path, out_dir: Path, dpi: int = 150) -> list[Path]:
    """
    Render each page of a PDF as a PNG.
    Returns list of image paths in page order.
    """
    logger.info("Opening PDF: %s", pdf_path.name)
    doc  = fitz.open(str(pdf_path))
    zoom = dpi / 72
    mat  = fitz.Matrix(zoom, zoom)
    image_paths = []

    for page_num in range(len(doc)):
        page     = doc.load_page(page_num)
        pix      = page.get_pixmap(matrix=mat, alpha=False)
        out_name = f"{pdf_path.stem}_page_{page_num + 1}.png"
        out_path = out_dir / out_name
        pix.save(str(out_path))
        image_paths.append(out_path)
        logger.debug("  Rendered page %d → %s (%dx%d px)",
                     page_num + 1, out_name, pix.width, pix.height)

    doc.close()
    logger.info("Done: %d page(s) from %s", len(image_paths), pdf_path.name)
    return image_paths

In [ ]:
# Process all PDFs
pdf_files = sorted(INPUT_FOLDER.glob("*.pdf"))
logger.info("Found %d PDF(s) in input_folder", len(pdf_files))

pdf_image_map: dict[str, list[Path]] = {}

for pdf_path in pdf_files:
    images = pdf_to_images(pdf_path, TEMP_FOLDER, dpi=PAGE_DPI)
    pdf_image_map[pdf_path.stem] = images

total = sum(len(v) for v in pdf_image_map.values())
logger.info("Total pages converted: %d", total)

In [ ]:
# Preview first page of first PDF
from PIL import Image
import matplotlib.pyplot as plt

if pdf_image_map:
    first_stem  = next(iter(pdf_image_map))
    first_image = pdf_image_map[first_stem][0]
    logger.info("Previewing: %s", first_image.name)

    img = Image.open(first_image)
    plt.figure(figsize=(8, 11))
    plt.imshow(img, cmap="gray")
    plt.axis("off")
    plt.title(f"{first_image.name}  ({img.width}x{img.height} px)")
    plt.tight_layout()
    plt.show()
else:
    logger.warning("No PDFs found — add files to input_folder/ and re-run.")